In [1]:
# Cell 1 - Environment check
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'WARNING: No GPU detected -- training will be slow on CPU')

Mon Jun 22 21:46:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:41:00.0 Off |                    0 |
| N/A   33C    P0             36W /  250W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Cell 2 - Configuration (10M, TinyShakespeare, ResidualGPT)
import torch, torch.nn as nn, torch.nn.functional as F
import math, json, time, os, random
import numpy as np
from pathlib import Path
from dataclasses import dataclass, asdict
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
PROJECT_ID = 'residual_gpt_10m_reproduction'
OUT_DIR = (Path('/workspace') if Path('/workspace').exists() else Path('.')) / PROJECT_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = OUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
# Model hyperparameters -- EXACT paper Table 1, 10M column
N_LAYER    = 6
N_EMBD     = 384
N_HEAD     = 6
BLOCK_SIZE = 256
DROPOUT    = 0.2
# Training hyperparameters -- EXACT paper Table 1, 10M column
BATCH_SIZE      = 64
GRAD_ACCUM      = 1
MAX_ITERS       = 3000
EVAL_INTERVAL   = 100
EVAL_BATCHES    = 50
LEARNING_RATE   = 3e-4   # CONSTANT at 10M -- no warmup, no decay (paper Table 1)
WEIGHT_DECAY    = 0.1
BETA1, BETA2    = 0.9, 0.95
GRAD_CLIP       = 1.0
# The four configurations under test
CONFIGS = {
    'FullResidual_std': dict(use_attention_skip=True,  use_mlp_skip=True),
    'NoResidual_std':   dict(use_attention_skip=False, use_mlp_skip=False),
    'AttnOnly_std':     dict(use_attention_skip=True,  use_mlp_skip=False),
    'FFNOnly_std':      dict(use_attention_skip=False, use_mlp_skip=True),
}
# Seeds: 1337 for the base configs, all 3 seeds for the stability check
BASE_SEED = 1337
STABILITY_SEEDS = [1337, 42, 123]  # used for AttnOnly_std and FFNOnly_std
print(f'Project: {PROJECT_ID}')
print(f'Configs: {list(CONFIGS.keys())}')
print(f'lr = {LEARNING_RATE} (constant, no schedule)')

Device: cuda
Project: residual_gpt_10m_reproduction
Configs: ['FullResidual_std', 'NoResidual_std', 'AttnOnly_std', 'FFNOnly_std']
lr = 0.0003 (constant, no schedule)


In [3]:
# Cell 3 - Dataset (TinyShakespeare, character-level, vocab=65)
import urllib.request
DATA_DIR = OUT_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
txt_path = DATA_DIR / 'input.txt'
if not txt_path.exists():
    print('Downloading TinyShakespeare...')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        txt_path
    )
text = txt_path.read_text()
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f'Vocab size: {VOCAB_SIZE}  (paper states 65 -- {"MATCH" if VOCAB_SIZE==65 else "MISMATCH, check source text"})')
print(f'Train tokens: {len(train_data):,}  (paper states ~1.0M)')
print(f'Val tokens:   {len(val_data):,}    (paper states ~111K)')

Vocab size: 65  (paper states 65 -- MATCH)
Train tokens: 1,003,854  (paper states ~1.0M)
Val tokens:   111,540    (paper states ~111K)


In [4]:
# Cell 4 - Batcher with FIXED validation batches (the critical ResidualGPT fix)

class SequenceBatcher:
    def __init__(self, train_data, val_data, block_size, batch_size, device,
                 fixed_val_seed=99991, n_fixed_val_batches=50):
        self.train_data = train_data
        self.val_data   = val_data
        self.block_size = block_size
        self.batch_size = batch_size
        self.device     = device
        # Build fixed val batches ONCE using an isolated generator, so this
        # never perturbs the global / per-experiment RNG state used for
        # training-batch sampling.
        g = torch.Generator().manual_seed(fixed_val_seed)
        self.fixed_val_batches = []
        max_start = len(val_data) - block_size - 1
        for _ in range(n_fixed_val_batches):
            ix = torch.randint(max_start, (batch_size,), generator=g)
            xb = torch.stack([val_data[i:i+block_size]     for i in ix]).to(device)
            yb = torch.stack([val_data[i+1:i+block_size+1] for i in ix]).to(device)
            self.fixed_val_batches.append((xb, yb))
    def sample(self, split):
        d = self.train_data if split == 'train' else self.val_data
        max_start = len(d) - self.block_size - 1
        ix = torch.randint(max_start, (self.batch_size,))
        xb = torch.stack([d[i:i+self.block_size]     for i in ix]).to(self.device)
        yb = torch.stack([d[i+1:i+self.block_size+1] for i in ix]).to(self.device)
        return xb, yb
# Built ONCE -- shared across every config and every seed, so all comparisons
# in this notebook evaluate on identical validation data.
BATCHER = SequenceBatcher(train_data, val_data, BLOCK_SIZE, BATCH_SIZE, DEVICE)
print(f'Fixed val batches built: {len(BATCHER.fixed_val_batches)} batches, seed=99991')

Fixed val batches built: 50 batches, seed=99991


In [5]:
# Cell 5 - ResidualGPT model
#
# Two correctness requirements baked in here, both confirmed against the
# paper's described architecture:
#   (a) NO runtime gain multiplier inside forward(). Gain is a weight-init-
#       time effect only, applied via apply_output_gains() before the
#       optimizer is constructed.
#   (b) Weight tying between token_embed and lm_head.
@dataclass
class ResidualSwitch:
    use_attention_skip: bool = True
    use_mlp_skip:       bool = True
    attention_gain:     float = 1.0
    mlp_gain:           float = 1.0
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert N_EMBD % N_HEAD == 0
        self.qkv  = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.out  = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)
        self.n_head = N_HEAD
        self.head_dim = N_EMBD // N_HEAD
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.drop.p if self.training else 0.0, is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.out(y))
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(N_EMBD, 4 * N_EMBD, bias=False)
        self.fc2  = nn.Linear(4 * N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))
class Block(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch = switch
        self.ln1    = nn.LayerNorm(N_EMBD)
        self.ln2    = nn.LayerNorm(N_EMBD)
        self.attn   = CausalSelfAttention()
        self.mlp    = MLP()
    def forward(self, x):
        # NO runtime multiplier here -- gain is a weight-init-time effect only.
        a = self.attn(self.ln1(x))
        x = x + a if self.switch.use_attention_skip else a
        m = self.mlp(self.ln2(x))
        x = x + m if self.switch.use_mlp_skip else m
        return x
class ResidualGPT(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch      = switch
        self.token_embed = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_embed   = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop        = nn.Dropout(DROPOUT)
        self.blocks      = nn.ModuleList([Block(switch) for _ in range(N_LAYER)])
        self.ln_f        = nn.LayerNorm(N_EMBD)
        self.lm_head     = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)
        # Weight tying
        self.lm_head.weight = self.token_embed.weight
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def apply_output_gains(self):
        """Weight-level gain, applied ONCE before optimizer construction.
        With gain=1.0 (every config in this notebook) this is a no-op,
        included only so the code path exists and is correct if gain != 1.0
        configs are added later."""
        if self.switch.mlp_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.mlp.fc2.weight.mul_(self.switch.mlp_gain)
        if self.switch.attention_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.attn.out.weight.mul_(self.switch.attention_gain)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.token_embed(idx) + self.pos_embed(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss
    def param_count(self):
        return sum(p.numel() for p in self.parameters())
print('Model class defined')

Model class defined


In [6]:
# Cell 6 - Training and evaluation helpers
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
@torch.no_grad()
def estimate_val_loss(model, batcher, n_batches):
    model.eval()
    losses = []
    for xb, yb in batcher.fixed_val_batches[:n_batches]:
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))
@torch.no_grad()
def estimate_train_loss(model, batcher, n_batches):
    model.eval()
    losses = []
    for _ in range(n_batches):
        xb, yb = batcher.sample('train')
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))
def make_optimizer(model):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (decay if p.ndim >= 2 else no_decay).append(p)
    return torch.optim.AdamW(
        [{'params': decay, 'weight_decay': WEIGHT_DECAY},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=LEARNING_RATE, betas=(BETA1, BETA2)
    )
def train_one_run(name, seed, switch_kwargs, max_iters=MAX_ITERS,
                   eval_interval=EVAL_INTERVAL, eval_batches=EVAL_BATCHES,
                   verbose=True):
    set_seed(seed)
    switch = ResidualSwitch(**switch_kwargs)
    model = ResidualGPT(switch).to(DEVICE)
    model.apply_output_gains()
    optimizer = make_optimizer(model)
    history = []
    best_val = float('inf')
    best_step = -1
    started = time.time()
    val0 = estimate_val_loss(model, BATCHER, eval_batches)
    train0 = estimate_train_loss(model, BATCHER, eval_batches)
    history.append({'step': 0, 'train_loss': train0, 'val_loss': val0})
    if val0 < best_val:
        best_val, best_step = val0, 0
    # NOTE: LEARNING_RATE is CONSTANT at 10M scale per paper Table 1 --
    # no warmup, no decay. Do not apply a schedule here.
    for group in optimizer.param_groups:
        group['lr'] = LEARNING_RATE
    for step in range(1, max_iters + 1):
        optimizer.zero_grad(set_to_none=True)
        xb, yb = BATCHER.sample('train')
        _, loss = model(xb, yb)
        loss.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        if step % eval_interval == 0 or step == max_iters:
            val_loss = estimate_val_loss(model, BATCHER, eval_batches)
            train_loss = estimate_train_loss(model, BATCHER, eval_batches)
            is_best = val_loss < best_val
            if is_best:
                best_val, best_step = val_loss, step
            history.append({'step': step, 'train_loss': train_loss, 'val_loss': val_loss})
            if verbose:
                elapsed = time.time() - started
                print(f'  {name} seed={seed} step={step:5d} train={train_loss:.4f} '
                      f'val={val_loss:.4f} best={best_val:.4f}@{best_step} ({elapsed:.0f}s)')
    return {
        'name': name,
        'seed': seed,
        **switch_kwargs,
        'best_step': best_step,
        'best_val_loss': best_val,
        'best_val_ppl': math.exp(min(best_val, 20)),
        'final_step': max_iters,
        'final_train_loss': history[-1]['train_loss'],
        'final_val_loss': history[-1]['val_loss'],
        'final_val_ppl': math.exp(min(history[-1]['val_loss'], 20)),
        'history': history,
        'param_count': model.param_count(),
    }
print('Training functions ready')

Training functions ready


In [7]:
# Cell 7 - Checkpoint helpers (save/skip-if-exists, so this notebook is safely re-runnable)
def result_path(name, seed):
    return CKPT_DIR / f'{name}_seed{seed}.json'
def save_result(name, seed, result):
    path = result_path(name, seed)
    with open(path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'Saved: {path}')
def load_result_if_exists(name, seed):
    path = result_path(name, seed)
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None
print('Checkpoint helpers ready')

Checkpoint helpers ready


In [8]:
# Cell 8 - SANITY CHECK: quick partial run before committing to the full experiment
#
# Trains FullResidual_std for only 300 steps (10% of the full run) and checks
# the val loss is descending. This is NOT meant to match the final 1.5017
# number -- it's meant to catch gross architecture problems (flat/diverging
# loss) before spending time on the full 3,000-step runs below.
print('Running sanity check: FullResidual_std, seed=1337, 300 steps only...')
sanity = train_one_run('FullResidual_sanity', 1337, CONFIGS['FullResidual_std'],
                        max_iters=300, eval_interval=100, verbose=True)
print()
print(f'Sanity check val_loss @ step 300: {sanity["final_val_loss"]:.4f}')
print('PASS CRITERIA: should be descending and well below the random-init')
print('level (~4.17, i.e. ln(65)) by step 300. If flat or still near 4.0+,')
print('stop and check the architecture before running the full experiment.')

Running sanity check: FullResidual_std, seed=1337, 300 steps only...
  FullResidual_sanity seed=1337 step=  100 train=2.4607 val=2.4765 best=2.4765@100 (14s)
  FullResidual_sanity seed=1337 step=  200 train=2.3788 val=2.4083 best=2.4083@200 (26s)
  FullResidual_sanity seed=1337 step=  300 train=2.2085 val=2.2569 best=2.2569@300 (37s)

Sanity check val_loss @ step 300: 2.2569
PASS CRITERIA: should be descending and well below the random-init
level (~4.17, i.e. ln(65)) by step 300. If flat or still near 4.0+,
stop and check the architecture before running the full experiment.


In [9]:
# Cell 9 - THE FULL EXPERIMENT
#
# Runs all 4 base configs at seed 1337, plus AttnOnly_std and FFNOnly_std
# again at seeds 42 and 123 for the 3-seed stability claim in paper Table 3.
#
# Only run this after the sanity check above looks correct.
results = {}
# Base configs, seed 1337 only
for name, switch_kwargs in CONFIGS.items():
    seed = BASE_SEED
    existing = load_result_if_exists(name, seed)
    if existing is not None:
        print(f'Skipping {name} (seed={seed}) -- already completed (best_val={existing["best_val_loss"]:.4f})')
        results[(name, seed)] = existing
        continue
    print(f'\n{"="*70}\nRunning: {name} (seed={seed})\n{"="*70}')
    result = train_one_run(name, seed, switch_kwargs)
    save_result(name, seed, result)
    results[(name, seed)] = result
# Stability seeds for AttnOnly_std and FFNOnly_std only
for name in ['AttnOnly_std', 'FFNOnly_std']:
    for seed in STABILITY_SEEDS:
        if seed == BASE_SEED:
            continue  # already run above
        existing = load_result_if_exists(name, seed)
        if existing is not None:
            print(f'Skipping {name} (seed={seed}) -- already completed (best_val={existing["best_val_loss"]:.4f})')
            results[(name, seed)] = existing
            continue
        print(f'\n{"="*70}\nRunning: {name} (seed={seed})\n{"="*70}')
        result = train_one_run(name, seed, CONFIGS[name])
        result['seed'] = seed
        save_result(name, seed, result)
        results[(name, seed)] = result
print('\nAll requested runs complete.')


Running: FullResidual_std (seed=1337)
  FullResidual_std seed=1337 step=  100 train=2.4607 val=2.4765 best=2.4765@100 (14s)
  FullResidual_std seed=1337 step=  200 train=2.3788 val=2.4083 best=2.4083@200 (25s)
  FullResidual_std seed=1337 step=  300 train=2.2085 val=2.2569 best=2.2569@300 (37s)
  FullResidual_std seed=1337 step=  400 train=2.0548 val=2.1314 best=2.1314@400 (48s)
  FullResidual_std seed=1337 step=  500 train=1.9165 val=2.0249 best=2.0249@500 (60s)
  FullResidual_std seed=1337 step=  600 train=1.8117 val=1.9475 best=1.9475@600 (71s)
  FullResidual_std seed=1337 step=  700 train=1.7109 val=1.8727 best=1.8727@700 (83s)
  FullResidual_std seed=1337 step=  800 train=1.6543 val=1.8219 best=1.8219@800 (94s)
  FullResidual_std seed=1337 step=  900 train=1.5960 val=1.7713 best=1.7713@900 (106s)
  FullResidual_std seed=1337 step= 1000 train=1.5544 val=1.7501 best=1.7501@1000 (117s)
  FullResidual_std seed=1337 step= 1100 train=1.5042 val=1.7003 best=1.7003@1100 (129s)
  FullResi

In [18]:
# Cell 10 - Compare against the verified target values
TARGETS = {
    ('FullResidual_std', 1337): 1.5017,
    ('NoResidual_std',   1337): 3.3564,
    ('AttnOnly_std',     1337): 1.5800,
    ('AttnOnly_std',     42):   1.6040,
    ('AttnOnly_std',     123):  1.6157,
    ('FFNOnly_std',      1337): 3.3481,
    ('FFNOnly_std',      42):   3.3497,
    ('FFNOnly_std',      123):  3.3502,
}
print(f'{"Config":<20}{"Seed":>6}{"This run":>12}{"Target":>10}{"Diff":>10}')
print('-' * 58)
for (name, seed), target in TARGETS.items():
    r = results.get((name, seed))
    if r is None:
        print(f'{name:<20}{seed:>6}{"MISSING":>12}{target:>10.4f}{"--":>10}')
        continue
    val = r['best_val_loss']
    diff = val - target
    print(f'{name:<20}{seed:>6}{val:>12.4f}{target:>10.4f}{diff:>+10.4f}')
print()
print('A small diff is normal seed/hardware/library-version noise.')
print('A large diff suggests this reimplementation diverges architecturally')


Config                Seed    This run    Target      Diff
----------------------------------------------------------
FullResidual_std      1337      1.4863    1.5017   -0.0154
NoResidual_std        1337      3.3560    3.3564   -0.0004
AttnOnly_std          1337      1.9683    1.5800   +0.3883
AttnOnly_std            42      3.3543    1.6040   +1.7503
AttnOnly_std           123      2.1313    1.6157   +0.5156
FFNOnly_std           1337      3.3499    3.3481   +0.0018
FFNOnly_std             42      3.3496    3.3497   -0.0001
FFNOnly_std            123      3.3497    3.3502   -0.0005

A small diff is normal seed/hardware/library-version noise.
A large diff suggests this reimplementation diverges architecturally


In [ ]:
# Cell 11 - Compute 3-seed mean/std for AttnOnly_std and FFNOnly_std (paper Table 3)
for name in ['AttnOnly_std', 'FFNOnly_std']:
    vals = []
    for seed in STABILITY_SEEDS:
        r = results.get((name, seed))
        if r is not None:
            vals.append(r['best_val_loss'])
    if len(vals) == 3:
        mean = float(np.mean(vals))
        std = float(np.std(vals))  # population std, ddof=0 -- matches paper convention
        print(f'{name}: {mean:.4f} +/- {std:.4f}  (n=3, population std)')
    else:
        print(f'{name}: incomplete ({len(vals)}/3 seeds) -- rerun Cell 9')
print()
print('Compare against paper Table 3:')
print('  AttnOnly_std: 1.5999 +/- 0.0149')
print('  FFNOnly_std:  3.3493 +/- 0.0009')

AttnOnly_std: 2.4846 +/- 0.6185  (n=3, population std)
FFNOnly_std: 3.3497 +/- 0.0001  (n=3, population std)

Compare against paper Table 3:
  AttnOnly_std: 1.5999 +/- 0.0149
  FFNOnly_std:  3.3493 +/- 0.0009


LocalMIxer Experiment (Section 7.4)


In [12]:
# Local mixer: same param budget as CausalSelfAttention, no cross-position mixing
class LocalMixer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(N_EMBD, 2 * N_EMBD, bias=False)
        self.fc2  = nn.Linear(2 * N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

p1 = sum(p.numel() for p in LocalMixer().parameters())
p2 = sum(p.numel() for p in CausalSelfAttention().parameters())
print(f'LocalMixer: {p1:,} params, CausalSelfAttention: {p2:,} params, ratio {p1/p2:.2f}')

LocalMixer: 589,824 params, CausalSelfAttention: 589,824 params, ratio 1.00


In [13]:
class LocalMixerBlock(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch = switch
        self.ln1    = nn.LayerNorm(N_EMBD)
        self.ln2    = nn.LayerNorm(N_EMBD)
        self.mixer  = LocalMixer()
        self.mlp    = MLP()
    def forward(self, x):
        a = self.mixer(self.ln1(x))
        x = x + a if self.switch.use_attention_skip else a
        m = self.mlp(self.ln2(x))
        x = x + m if self.switch.use_mlp_skip else m
        return x

class LocalMixerGPT(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch      = switch
        self.token_embed = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_embed   = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop        = nn.Dropout(DROPOUT)
        self.blocks      = nn.ModuleList([LocalMixerBlock(switch) for _ in range(N_LAYER)])
        self.ln_f        = nn.LayerNorm(N_EMBD)
        self.lm_head     = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)
        self.lm_head.weight = self.token_embed.weight
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def apply_output_gains(self):
        if self.switch.mlp_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.mlp.fc2.weight.mul_(self.switch.mlp_gain)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.token_embed(idx) + self.pos_embed(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss
    def param_count(self):
        return sum(p.numel() for p in self.parameters())

In [14]:
def train_one_local_mixer_run(name, seed, switch_kwargs, max_iters=MAX_ITERS,
                               eval_interval=EVAL_INTERVAL, eval_batches=EVAL_BATCHES,
                               verbose=True):
    set_seed(seed)
    switch = ResidualSwitch(**switch_kwargs)
    model = LocalMixerGPT(switch).to(DEVICE)
    model.apply_output_gains()
    optimizer = make_optimizer(model)
    history = []
    best_val = float('inf')
    best_step = -1
    started = time.time()
    val0 = estimate_val_loss(model, BATCHER, eval_batches)
    train0 = estimate_train_loss(model, BATCHER, eval_batches)
    history.append({'step': 0, 'train_loss': train0, 'val_loss': val0})
    if val0 < best_val:
        best_val, best_step = val0, 0
    for group in optimizer.param_groups:
        group['lr'] = LEARNING_RATE
    for step in range(1, max_iters + 1):
        optimizer.zero_grad(set_to_none=True)
        xb, yb = BATCHER.sample('train')
        _, loss = model(xb, yb)
        loss.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        if step % eval_interval == 0 or step == max_iters:
            val_loss = estimate_val_loss(model, BATCHER, eval_batches)
            train_loss = estimate_train_loss(model, BATCHER, eval_batches)
            if val_loss < best_val:
                best_val, best_step = val_loss, step
            history.append({'step': step, 'train_loss': train_loss, 'val_loss': val_loss})
            if verbose:
                elapsed = time.time() - started
                print(f'  {name} seed={seed} step={step:5d} train={train_loss:.4f} '
                      f'val={val_loss:.4f} best={best_val:.4f}@{best_step} ({elapsed:.0f}s)')
    return {
        'name': name,
        'seed': seed,
        **switch_kwargs,
        'best_step': best_step,
        'best_val_loss': best_val,
        'best_val_ppl': math.exp(min(best_val, 20)),
        'final_step': max_iters,
        'final_train_loss': history[-1]['train_loss'],
        'final_val_loss': history[-1]['val_loss'],
        'final_val_ppl': math.exp(min(history[-1]['val_loss'], 20)),
        'history': history,
        'param_count': model.param_count(),
    }

In [15]:
LOCAL_MIXER_CONFIG = dict(use_attention_skip=True, use_mlp_skip=False)

mixer_sanity = train_one_local_mixer_run(
    'AttnOnly_LocalMixer_sanity', 1337, LOCAL_MIXER_CONFIG,
    max_iters=300, eval_interval=100, verbose=True
)
print(f'val_loss @ step 300: {mixer_sanity["final_val_loss"]:.4f} (should be well below ln(65)=4.17)')

  AttnOnly_LocalMixer_sanity seed=1337 step=  100 train=2.5276 val=2.5234 best=2.5234@100 (13s)
  AttnOnly_LocalMixer_sanity seed=1337 step=  200 train=2.5038 val=2.5142 best=2.5142@200 (24s)
  AttnOnly_LocalMixer_sanity seed=1337 step=  300 train=2.4905 val=2.5087 best=2.5087@300 (35s)
val_loss @ step 300: 2.5087 (should be well below ln(65)=4.17)


In [16]:
# AttnOnly_LocalMixer_std, seed 1337
local_mixer_results = {}
name, seed = 'AttnOnly_LocalMixer_std', BASE_SEED
existing = load_result_if_exists(name, seed)
if existing is not None:
    print(f'Skipping {name} -- already done (best_val={existing["best_val_loss"]:.4f})')
    local_mixer_results[(name, seed)] = existing
else:
    result = train_one_local_mixer_run(name, seed, LOCAL_MIXER_CONFIG)
    save_result(name, seed, result)
    local_mixer_results[(name, seed)] = result

  AttnOnly_LocalMixer_std seed=1337 step=  100 train=2.5276 val=2.5234 best=2.5234@100 (13s)
  AttnOnly_LocalMixer_std seed=1337 step=  200 train=2.5038 val=2.5142 best=2.5142@200 (24s)
  AttnOnly_LocalMixer_std seed=1337 step=  300 train=2.4905 val=2.5087 best=2.5087@300 (35s)
  AttnOnly_LocalMixer_std seed=1337 step=  400 train=2.4829 val=2.5037 best=2.5037@400 (45s)
  AttnOnly_LocalMixer_std seed=1337 step=  500 train=2.4744 val=2.4985 best=2.4985@500 (56s)
  AttnOnly_LocalMixer_std seed=1337 step=  600 train=2.4760 val=2.5015 best=2.4985@500 (67s)
  AttnOnly_LocalMixer_std seed=1337 step=  700 train=2.4697 val=2.4960 best=2.4960@700 (78s)
  AttnOnly_LocalMixer_std seed=1337 step=  800 train=2.4705 val=2.4944 best=2.4944@800 (88s)
  AttnOnly_LocalMixer_std seed=1337 step=  900 train=2.4699 val=2.4982 best=2.4944@800 (99s)
  AttnOnly_LocalMixer_std seed=1337 step= 1000 train=2.4691 val=2.4941 best=2.4941@1000 (110s)
  AttnOnly_LocalMixer_std seed=1337 step= 1100 train=2.4678 val=2.49

In [17]:
r = local_mixer_results.get(('AttnOnly_LocalMixer_std', 1337))
val = r['best_val_loss']
print(f'AttnOnly_LocalMixer_std: {val:.4f}')
print(f'AttnOnly_std (attention): 1.5800')
print(f'FFNOnly_std (collapse):   3.3481')
print()
if val > 3.20:
    print('Collapses near the FFNOnly floor -- consistent with cross-position routing.')
elif val < 1.75:
    print('Recovers like attention despite no cross-position mixing -- routing hypothesis not supported.')
else:
    print('Partial recovery -- run seeds 42/123 before concluding.')

AttnOnly_LocalMixer_std: 2.4870
AttnOnly_std (attention): 1.5800
FFNOnly_std (collapse):   3.3481

Partial recovery -- run seeds 42/123 before concluding.
